# 30 feature schema · type별 allow/block LogisticRegression 학습 — v3

`logistic_30_v3.ipynb` — booking / captcha / detail 각 type 마다 allow/block 이진 분류 모델을 하나씩 학습한다 (총 3개 모델).

**v1과의 차이**
- v1: 전체 데이터를 하나의 allow/block 모델로 학습.
- v3: type 별로 데이터를 분리해서 type 별 모델 3개 학습.
- 학습 데이터에 `data_tickle_0515_human_v2` (모두 ALLOW) 추가됨 — `source_dataset` 컬럼으로 추적.

**공통**
- `BASE_FEATURES_30` 명시 고정 (RandomForest / LogisticRegression v3 노트북 양쪽 동일 순서 — 앙상블 호환).
- `ALLOW=0`, `BLOCK=1`. positive class = BLOCK. `P_BLOCK = predict_proba(X)[:, 1]`.
- 모델 저장 위치: `../model_joblib/logistic_30_v3/{type}/` 안에 `model.joblib` + `meta.json` + `metrics.json` 3종.

## 1. 설정 및 import

In [1]:
import os
import json
import time
import warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

args = SimpleNamespace(
    RUN_VERSION="logistic_30_v3",
    DATA_ROOT="../../data/train_data_0518/train_data_0518",
    MODEL_SAVE_DIR="../model_joblib/logistic_30_v3",
    INCLUDED_DATA_DIRS=["data_0515_1400_gt_v2", "data_tickle_0515_human_v2"],
    TYPES=["booking", "captcha", "detail"],
    VALID_SIZE=0.2,
    RANDOM_STATE=42,
    SAVE_MODEL=True,
    LR_C=1.0,
    LR_PENALTY="l2",
    LR_SOLVER="lbfgs",
    LR_CLASS_WEIGHT="balanced",
    LR_MAX_ITER=3000,
)
args

namespace(RUN_VERSION='logistic_30_v3',
          DATA_ROOT='../../data/train_data_0518/train_data_0518',
          MODEL_SAVE_DIR='../model_joblib/logistic_30_v3',
          INCLUDED_DATA_DIRS=['data_0515_1400_gt_v2',
                              'data_tickle_0515_human_v2'],
          TYPES=['booking', 'captcha', 'detail'],
          VALID_SIZE=0.2,
          RANDOM_STATE=42,
          SAVE_MODEL=True,
          LR_C=1.0,
          LR_PENALTY='l2',
          LR_SOLVER='lbfgs',
          LR_CLASS_WEIGHT='balanced',
          LR_MAX_ITER=3000)

## 2. BASE_FEATURES_30 정의

RandomForest / LogisticRegression v3 양쪽 노트북에서 **동일한 30개 feature 순서** 를 사용한다 (앙상블 시 입력 schema 호환).

`EXCLUDED_FEATURES` 는 30개 기준에 포함하지 않는 feature (4개).

In [2]:
BASE_FEATURES_30 = [
    "reclick_rate",
    "misclick_rate",
    "double_click_rate",
    "mouse_overshoot_flag",
    "mousemove_event_rate",
    "pre_click_scroll_flag",
    "time_to_first_click_ms",
    "mouse_acceleration_mean",
    "mouse_speed_change_mean",
    "pre_click_hover_time_ms",
    "mouse_stop_segment_count",
    "mouse_avg_speed_px_per_ms",
    "mouse_hover_dwell_time_ms",
    "mouse_max_speed_px_per_ms",
    "mouse_path_curvature_mean",
    "pre_click_mousemove_count",
    "click_position_repeat_rate",
    "mouse_direction_change_count",
    "mouse_path_straightness_score",
    "edge_or_fixed_point_visit_rate",
    "mouse_total_travel_distance_px",
    "click_sequence_consistency_score",
    "immediate_post_render_click_rate",
    "pre_click_path_300ms_straightness",
    "pre_click_path_500ms_straightness",
    "click_offset_from_element_center_px",
    "time_from_element_visible_to_click_ms",
    "pre_click_path_300ms_total_distance_px",
    "pre_click_path_500ms_total_distance_px",
    "time_from_element_clickable_to_click_ms",
]

EXCLUDED_FEATURES = [
    "mouse_jerk_mean",
    "inter_click_interval_ms",
    "click_offset_variance_px",
    "inter_element_move_interval_std_ms",
]

assert len(BASE_FEATURES_30) == 30, f"BASE_FEATURES_30 must be 30, got {len(BASE_FEATURES_30)}"
assert len(set(BASE_FEATURES_30)) == 30, "BASE_FEATURES_30 has duplicates"
print(f"BASE_FEATURES_30 length: {len(BASE_FEATURES_30)}")
print(f"EXCLUDED_FEATURES length: {len(EXCLUDED_FEATURES)}")

BASE_FEATURES_30 length: 30
EXCLUDED_FEATURES length: 4


## 3. 데이터 로딩 함수

데이터셋별 라벨 해석:
- `data_0515_1400_gt_v2/<type>/{allow,block}/` — 폴더명에서 type 과 label 추출 (`source_dataset="gt_v2"`).
- `data_tickle_0515_human_v2/<type>/...` — type 만 폴더명에서 추출. label 은 모두 `ALLOW` 로 강제 (`source_dataset="tickle_human_v2"`).

두 데이터셋은 물리적으로 합치지 않고 코드에서 concat. `source_dataset`, `source_file`, `source_path` 컬럼 유지.

In [3]:
TYPE_NAMES = ("booking", "captcha", "detail")
LABEL_NAMES_ALLOWED = ("allow", "block")


def _detect_type_from_parts(parts_lower, base_dir_lower):
    # base_dir 직계 하위 폴더명에서 type 을 찾는다.
    try:
        idx = parts_lower.index(base_dir_lower)
    except ValueError:
        return None
    if idx + 1 >= len(parts_lower):
        return None
    cand = parts_lower[idx + 1]
    return cand if cand in TYPE_NAMES else None


def load_dataset(root_dir: Path, base_dir_name: str, source_dataset: str,
                  default_label: str | None = None):
    """
    root_dir 아래의 모든 .json 을 읽어 DataFrame 으로 반환.
    - default_label=None  → 경로의 'allow'/'block' 폴더로 라벨 결정 (없으면 skip).
    - default_label='ALLOW' → 모든 샘플을 ALLOW 로 강제 (tickle_human_v2 처럼).
    부여 컬럼: type, label_norm, y, source_dataset, source_file, source_path.
    """
    root = Path(root_dir)
    if not root.exists():
        raise FileNotFoundError(f"dataset root not found: {root.resolve()}")

    base_dir_lower = base_dir_name.lower()
    rows = []
    n_total, n_skip = 0, 0
    for jp in sorted(root.rglob("*.json")):
        n_total += 1
        try:
            with open(jp, encoding="utf-8") as f:
                obj = json.load(f)
        except Exception as e:
            warnings.warn(f"skip {jp}: {e}")
            n_skip += 1
            continue
        parts_lower = [p.lower() for p in jp.parts]
        t = _detect_type_from_parts(parts_lower, base_dir_lower)
        if t is None:
            warnings.warn(f"skip {jp}: cannot infer type from path")
            n_skip += 1
            continue
        if default_label is None:
            label_hits = [l for l in LABEL_NAMES_ALLOWED if l in parts_lower]
            if len(label_hits) != 1:
                warnings.warn(f"skip {jp}: cannot infer label (hits={label_hits})")
                n_skip += 1
                continue
            label_norm = label_hits[0].upper()
        else:
            label_norm = default_label
        flat = pd.json_normalize(obj, sep=".")
        flat["type"] = t
        flat["label_norm"] = label_norm
        flat["y"] = 0 if label_norm == "ALLOW" else 1
        flat["source_dataset"] = source_dataset
        flat["source_file"] = jp.name
        flat["source_path"] = str(jp)
        rows.append(flat)
    df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    return df, n_total, n_skip

## 4. 데이터 구조 탐색 및 포함 데이터 확인

두 데이터셋 폴더 존재 확인 + 폴더별 .json 파일 수 출력.

In [4]:
DATA_ROOT = Path(args.DATA_ROOT)
print(f"DATA_ROOT: {DATA_ROOT.resolve()}")
print(f"exists: {DATA_ROOT.exists()}")

for d in args.INCLUDED_DATA_DIRS:
    sub = DATA_ROOT / d
    print(f"  - {d}: exists={sub.exists()}")
    if sub.exists():
        n = sum(1 for _ in sub.rglob("*.json"))
        print(f"      total .json files: {n}")

DATA_ROOT: C:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\data\train_data_0518\train_data_0518
exists: True
  - data_0515_1400_gt_v2: exists=True
      total .json files: 773
  - data_tickle_0515_human_v2: exists=True
      total .json files: 581


## 5. source_dataset 컬럼 추가 및 전체 데이터 concat

두 데이터셋을 각각 읽어 source_dataset 컬럼을 부여한 뒤 하나의 DataFrame 으로 합친다.
- `data_0515_1400_gt_v2` → `source_dataset="gt_v2"`, label 은 allow/block 폴더로 결정
- `data_tickle_0515_human_v2` → `source_dataset="tickle_human_v2"`, label 은 모두 ALLOW 강제

In [5]:
df_gt, n_gt_total, n_gt_skip = load_dataset(
    DATA_ROOT / "data_0515_1400_gt_v2",
    base_dir_name="data_0515_1400_gt_v2",
    source_dataset="gt_v2",
    default_label=None,
)
print(f"[gt_v2] loaded={len(df_gt)}, json_files={n_gt_total}, skipped={n_gt_skip}")

df_tk, n_tk_total, n_tk_skip = load_dataset(
    DATA_ROOT / "data_tickle_0515_human_v2",
    base_dir_name="data_tickle_0515_human_v2",
    source_dataset="tickle_human_v2",
    default_label="ALLOW",
)
print(f"[tickle_human_v2] loaded={len(df_tk)}, json_files={n_tk_total}, skipped={n_tk_skip}")

df_all = pd.concat([df_gt, df_tk], ignore_index=True)
print(f"[concat] total={len(df_all)}")
print("\n[source_dataset x label_norm]")
print(df_all.groupby(["source_dataset", "label_norm"]).size().unstack(fill_value=0))
print("\n[source_dataset x type]")
print(df_all.groupby(["source_dataset", "type"]).size().unstack(fill_value=0))

C:\Users\SSAFY\AppData\Local\Temp\ipykernel_33664\474276336.py:64: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


[gt_v2] loaded=773, json_files=773, skipped=0


C:\Users\SSAFY\AppData\Local\Temp\ipykernel_33664\474276336.py:64: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


[tickle_human_v2] loaded=581, json_files=581, skipped=0
[concat] total=1354

[source_dataset x label_norm]
label_norm       ALLOW  BLOCK
source_dataset               
gt_v2               99    674
tickle_human_v2    581      0

[source_dataset x type]
type             booking  captcha  detail
source_dataset                           
gt_v2                250      260     263
tickle_human_v2      151      207     223


## 6. feature 컬럼 매핑

각 BASE_FEATURES_30 에 대해 df 의 실제 컬럼 (`<feature>` 또는 `metrics.<feature>`) 을 찾아 `feature_col_map` 구성. 누락 시 즉시 에러.

In [6]:
feature_col_map: dict[str, str] = {}
missing_features: list[str] = []

for f in BASE_FEATURES_30:
    if f in df_all.columns:
        feature_col_map[f] = f
    elif f"metrics.{f}" in df_all.columns:
        feature_col_map[f] = f"metrics.{f}"
    else:
        missing_features.append(f)

if missing_features:
    raise ValueError(f"Required features missing in df: {missing_features}")

selected_features_for_training = [feature_col_map[f] for f in BASE_FEATURES_30]

# 사용되지 않는 metrics.* numeric 컬럼 점검 (참고용)
metrics_cols = [c for c in df_all.columns if c.startswith("metrics.")]
used_set = set(selected_features_for_training)
extra_numeric_features_not_used = [c for c in metrics_cols if c not in used_set]

assert len(BASE_FEATURES_30) == 30
assert len(selected_features_for_training) == 30
assert list(feature_col_map.keys()) == BASE_FEATURES_30

print("BASE_FEATURES_30 length:", len(BASE_FEATURES_30))
print("selected_features_for_training length:", len(selected_features_for_training))
print("selected_features_for_training:", selected_features_for_training)
print("feature_names_for_save:", BASE_FEATURES_30)
print("excluded_features:", EXCLUDED_FEATURES)
print("missing features:", missing_features)
print("extra numeric features not used:", extra_numeric_features_not_used)

BASE_FEATURES_30 length: 30
selected_features_for_training length: 30
selected_features_for_training: ['metrics.reclick_rate', 'metrics.misclick_rate', 'metrics.double_click_rate', 'metrics.mouse_overshoot_flag', 'metrics.mousemove_event_rate', 'metrics.pre_click_scroll_flag', 'metrics.time_to_first_click_ms', 'metrics.mouse_acceleration_mean', 'metrics.mouse_speed_change_mean', 'metrics.pre_click_hover_time_ms', 'metrics.mouse_stop_segment_count', 'metrics.mouse_avg_speed_px_per_ms', 'metrics.mouse_hover_dwell_time_ms', 'metrics.mouse_max_speed_px_per_ms', 'metrics.mouse_path_curvature_mean', 'metrics.pre_click_mousemove_count', 'metrics.click_position_repeat_rate', 'metrics.mouse_direction_change_count', 'metrics.mouse_path_straightness_score', 'metrics.edge_or_fixed_point_visit_rate', 'metrics.mouse_total_travel_distance_px', 'metrics.click_sequence_consistency_score', 'metrics.immediate_post_render_click_rate', 'metrics.pre_click_path_300ms_straightness', 'metrics.pre_click_path_50

## 7. type별 데이터셋 생성

각 type 별로 데이터를 분리하고, **전체 concat 결과 안에서** allow/block 모두 존재하는지 확인.
(tickle_human_v2 단독은 allow 만 있는 것이 정상)

In [7]:
df_by_type: dict[str, pd.DataFrame] = {}
for t in args.TYPES:
    sub = df_all[df_all["type"] == t].reset_index(drop=True).copy()
    labels_present = set(sub["label_norm"].unique())
    if not {"ALLOW", "BLOCK"}.issubset(labels_present):
        raise ValueError(
            f"{t} dataset must contain both ALLOW and BLOCK labels. "
            f"got: {sorted(labels_present)}"
        )
    df_by_type[t] = sub
    print(f"[{t}] n={len(sub)} | source_dataset value_counts:")
    print(sub["source_dataset"].value_counts().to_string())
    print("  label_norm value_counts:", sub["label_norm"].value_counts().to_dict())

[booking] n=401 | source_dataset value_counts:
source_dataset
gt_v2              250
tickle_human_v2    151
  label_norm value_counts: {'BLOCK': 220, 'ALLOW': 181}
[captcha] n=467 | source_dataset value_counts:
source_dataset
gt_v2              260
tickle_human_v2    207
  label_norm value_counts: {'ALLOW': 241, 'BLOCK': 226}
[detail] n=486 | source_dataset value_counts:
source_dataset
gt_v2              263
tickle_human_v2    223
  label_norm value_counts: {'ALLOW': 258, 'BLOCK': 228}


## 8. type별 label/source 분포 + missing rate 확인

- type 별 30개 feature 의 missing rate 출력
- source_dataset 별 missing rate 도 함께 출력 (gt_v2 vs tickle_human_v2 분포 비교)

In [8]:
for t in args.TYPES:
    sub = df_by_type[t]
    X_full = sub[selected_features_for_training].copy()
    X_full.columns = BASE_FEATURES_30

    print(f"\n========== [{t}] missing rate (overall) ==========")
    mr_all = X_full.isna().mean().sort_values(ascending=False)
    print(mr_all.round(4).to_string())

    for sd in sorted(sub["source_dataset"].unique()):
        sub_sd = sub[sub["source_dataset"] == sd]
        X_sd = sub_sd[selected_features_for_training].copy()
        X_sd.columns = BASE_FEATURES_30
        mr = X_sd.isna().mean().sort_values(ascending=False)
        print(f"\n---- [{t} / {sd}] n={len(sub_sd)} missing rate ----")
        print(mr.round(4).to_string())


========== [booking] missing rate (overall) ==========
reclick_rate                               0.0
misclick_rate                              0.0
double_click_rate                          0.0
mouse_overshoot_flag                       0.0
mousemove_event_rate                       0.0
pre_click_scroll_flag                      0.0
time_to_first_click_ms                     0.0
mouse_acceleration_mean                    0.0
mouse_speed_change_mean                    0.0
pre_click_hover_time_ms                    0.0
mouse_stop_segment_count                   0.0
mouse_avg_speed_px_per_ms                  0.0
mouse_hover_dwell_time_ms                  0.0
mouse_max_speed_px_per_ms                  0.0
mouse_path_curvature_mean                  0.0
pre_click_mousemove_count                  0.0
click_position_repeat_rate                 0.0
mouse_direction_change_count               0.0
mouse_path_straightness_score              0.0
edge_or_fixed_point_visit_rate             0.0
mous

## 9. type별 train/valid split

우선순위:
1. group key (`trialId` → 없으면 `source_file`) 가 있으면 `GroupShuffleSplit` (`split_method="group_split"`)
2. group key 없으면 `train_test_split(stratify=label+source_dataset)` (`split_method="stratified_random_split"`)
3. 조합별 샘플 부족하면 label 만 stratify 로 fallback (`split_method="stratified_random_split_label_only"`)

In [9]:
GROUP_KEY_CANDIDATES = ["trialId", "trial_id", "session_id", "user_id", "file_id", "source_file", "source_path"]


def pick_group_key(df: pd.DataFrame) -> str | None:
    for k in GROUP_KEY_CANDIDATES:
        if k in df.columns and df[k].notna().all() and df[k].nunique() >= 2:
            return k
    return None


def split_type(sub: pd.DataFrame, valid_size: float, random_state: int):
    """
    Returns: train_idx, valid_idx, split_method, group_key
    train_idx / valid_idx 는 sub.index 기준 (sub.reset_index 된 상태이므로 0..N-1).
    """
    n = len(sub)
    y = sub["y"].values
    src = sub["source_dataset"].values
    group_key = pick_group_key(sub)

    if group_key is not None:
        groups = sub[group_key].astype(str).values
        # group split 이 valid 에 어떤 라벨도 가지 않을 위험이 있으니 한 번 검사
        gss = GroupShuffleSplit(n_splits=1, test_size=valid_size, random_state=random_state)
        tr_idx, va_idx = next(gss.split(np.zeros(n), y, groups))
        if len(set(y[va_idx])) >= 2:
            return tr_idx, va_idx, "group_split", group_key
        warnings.warn(f"group_split produced single-class valid set; falling back to stratified split")

    # try stratify by label + source_dataset
    strat_combo = pd.Series([f"{a}_{b}" for a, b in zip(y, src)])
    if strat_combo.value_counts().min() >= 2:
        tr_idx, va_idx = train_test_split(
            np.arange(n),
            test_size=valid_size,
            random_state=random_state,
            stratify=strat_combo,
        )
        return tr_idx, va_idx, "stratified_random_split", None

    # fallback: label 만 stratify
    tr_idx, va_idx = train_test_split(
        np.arange(n),
        test_size=valid_size,
        random_state=random_state,
        stratify=y,
    )
    return tr_idx, va_idx, "stratified_random_split_label_only", None


split_info: dict[str, dict] = {}
for t in args.TYPES:
    sub = df_by_type[t]
    tr_idx, va_idx, method, gkey = split_type(sub, args.VALID_SIZE, args.RANDOM_STATE)
    info = {
        "train_idx": tr_idx,
        "valid_idx": va_idx,
        "split_method": method,
        "group_key": gkey,
    }
    split_info[t] = info

    tr = sub.iloc[tr_idx]
    va = sub.iloc[va_idx]
    print(f"\n========== [{t}] ==========")
    print(f"  total={len(sub)}  train={len(tr)}  valid={len(va)}")
    print(f"  split_method={method}  group_key={gkey}")
    print(f"  total label dist: {sub['label_norm'].value_counts().to_dict()}")
    print(f"  train label dist: {tr['label_norm'].value_counts().to_dict()}")
    print(f"  valid label dist: {va['label_norm'].value_counts().to_dict()}")
    print(f"  total source x label:\n{sub.groupby(['source_dataset','label_norm']).size().unstack(fill_value=0)}")
    print(f"  train source x label:\n{tr.groupby(['source_dataset','label_norm']).size().unstack(fill_value=0)}")
    print(f"  valid source x label:\n{va.groupby(['source_dataset','label_norm']).size().unstack(fill_value=0)}")


========== [booking] ==========
  total=401  train=321  valid=80
  split_method=group_split  group_key=trialId
  total label dist: {'BLOCK': 220, 'ALLOW': 181}
  train label dist: {'BLOCK': 177, 'ALLOW': 144}
  valid label dist: {'BLOCK': 43, 'ALLOW': 37}
  total source x label:
label_norm       ALLOW  BLOCK
source_dataset               
gt_v2               30    220
tickle_human_v2    151      0
  train source x label:
label_norm       ALLOW  BLOCK
source_dataset               
gt_v2               23    177
tickle_human_v2    121      0
  valid source x label:
label_norm       ALLOW  BLOCK
source_dataset               
gt_v2                7     43
tickle_human_v2     30      0

========== [captcha] ==========
  total=467  train=377  valid=90
  split_method=group_split  group_key=trialId
  total label dist: {'ALLOW': 241, 'BLOCK': 226}
  train label dist: {'ALLOW': 199, 'BLOCK': 178}
  valid label dist: {'BLOCK': 48, 'ALLOW': 42}
  total source x label:
label_norm       ALLOW  BLOCK


## 10. type별 모델 학습 (LogisticRegression)

각 type 별로 동일 BASE_FEATURES_30 입력 순서, 동일 hyperparameter 의 LogisticRegression 모델 학습.

In [10]:
models: dict[str, Pipeline] = {}
train_data: dict[str, dict] = {}

for t in args.TYPES:
    sub = df_by_type[t]
    tr_idx = split_info[t]["train_idx"]
    va_idx = split_info[t]["valid_idx"]

    X = sub[selected_features_for_training].copy()
    X.columns = BASE_FEATURES_30   # 학습 시점부터 prefix 없는 이름으로 통일
    y = sub["y"].values

    X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X.iloc[va_idx], y[va_idx]

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            C=args.LR_C,
            penalty=args.LR_PENALTY,
            solver=args.LR_SOLVER,
            class_weight=args.LR_CLASS_WEIGHT,
            max_iter=args.LR_MAX_ITER,
            random_state=args.RANDOM_STATE,
        )),
    ])
    pipe.fit(X_tr, y_tr)
    models[t] = pipe
    train_data[t] = {
        "X_tr": X_tr, "y_tr": y_tr,
        "X_va": X_va, "y_va": y_va,
        "sub_tr": sub.iloc[tr_idx].reset_index(drop=True),
        "sub_va": sub.iloc[va_idx].reset_index(drop=True),
    }
    print(f"[{t}] fit done  train={len(X_tr)}  valid={len(X_va)}")

[booking] fit done  train=321  valid=80
[captcha] fit done  train=377  valid=90
[detail] fit done  train=391  valid=95


## 11. type별 기본 평가 (threshold=0.5)

validation set 기준 accuracy / precision / recall / f1 / roc_auc / confusion matrix / classification_report 출력.

In [11]:
def compute_basic_metrics(y_true, y_proba_block, threshold=0.5):
    y_pred = (y_proba_block >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    out = {
        "threshold": float(threshold),
        "n_samples": int(len(y_true)),
        "n_allow": int((y_true == 0).sum()),
        "n_block": int((y_true == 1).sum()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "tn": int(cm[0, 0]), "fp": int(cm[0, 1]),
        "fn": int(cm[1, 0]), "tp": int(cm[1, 1]),
        "pred_allow_count": int((y_pred == 0).sum()),
        "pred_block_count": int((y_pred == 1).sum()),
    }
    try:
        out["roc_auc"] = float(roc_auc_score(y_true, y_proba_block))
    except ValueError:
        out["roc_auc"] = None
    return out


overall_metrics_by_type: dict[str, dict] = {}
proba_cache: dict[str, np.ndarray] = {}

for t in args.TYPES:
    pipe = models[t]
    X_va = train_data[t]["X_va"]
    y_va = train_data[t]["y_va"]
    p_block = pipe.predict_proba(X_va)[:, 1]
    proba_cache[t] = p_block
    m = compute_basic_metrics(y_va, p_block, threshold=0.5)
    overall_metrics_by_type[t] = m
    y_pred = (p_block >= 0.5).astype(int)

    print(f"\n========== [{t}] valid metrics @ threshold=0.5 ==========")
    for k, v in m.items():
        print(f"  {k}: {v}")
    print("\nclassification_report:")
    print(classification_report(y_va, y_pred, target_names=["ALLOW", "BLOCK"], digits=4, zero_division=0))


========== [booking] valid metrics @ threshold=0.5 ==========
  threshold: 0.5
  n_samples: 80
  n_allow: 37
  n_block: 43
  accuracy: 1.0
  precision: 1.0
  recall: 1.0
  f1: 1.0
  tn: 37
  fp: 0
  fn: 0
  tp: 43
  pred_allow_count: 37
  pred_block_count: 43
  roc_auc: 1.0

classification_report:
              precision    recall  f1-score   support

       ALLOW     1.0000    1.0000    1.0000        37
       BLOCK     1.0000    1.0000    1.0000        43

    accuracy                         1.0000        80
   macro avg     1.0000    1.0000    1.0000        80
weighted avg     1.0000    1.0000    1.0000        80


========== [captcha] valid metrics @ threshold=0.5 ==========
  threshold: 0.5
  n_samples: 90
  n_allow: 42
  n_block: 48
  accuracy: 1.0
  precision: 1.0
  recall: 1.0
  f1: 1.0
  tn: 42
  fp: 0
  fn: 0
  tp: 48
  pred_allow_count: 42
  pred_block_count: 48
  roc_auc: 1.0

classification_report:
              precision    recall  f1-score   support

       ALLOW     1

## 12. type별 threshold sweep

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9] 각각에 대해 metric 계산.
best_threshold_by_f1 도 별도 기록.

In [12]:
THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

threshold_sweep_by_type: dict[str, list[dict]] = {}
best_threshold_by_type: dict[str, float] = {}

for t in args.TYPES:
    y_va = train_data[t]["y_va"]
    p_block = proba_cache[t]
    sweep = []
    best_f1 = -1.0
    best_thr = 0.5
    for thr in THRESHOLDS:
        m = compute_basic_metrics(y_va, p_block, threshold=thr)
        sweep.append(m)
        if m["f1"] > best_f1:
            best_f1 = m["f1"]
            best_thr = thr
    threshold_sweep_by_type[t] = sweep
    best_threshold_by_type[t] = best_thr

    df_sweep = pd.DataFrame(sweep)[[
        "threshold", "precision", "recall", "f1",
        "pred_allow_count", "pred_block_count",
        "tn", "fp", "fn", "tp",
    ]]
    print(f"\n========== [{t}] threshold sweep ==========")
    print(df_sweep.to_string(index=False))
    print(f"  best_threshold_by_f1 = {best_thr}  (f1={best_f1:.4f})")


========== [booking] threshold sweep ==========
 threshold  precision  recall       f1  pred_allow_count  pred_block_count  tn  fp  fn  tp
       0.1   0.977273     1.0 0.988506                36                44  36   1   0  43
       0.2   1.000000     1.0 1.000000                37                43  37   0   0  43
       0.3   1.000000     1.0 1.000000                37                43  37   0   0  43
       0.4   1.000000     1.0 1.000000                37                43  37   0   0  43
       0.5   1.000000     1.0 1.000000                37                43  37   0   0  43
       0.6   1.000000     1.0 1.000000                37                43  37   0   0  43
       0.7   1.000000     1.0 1.000000                37                43  37   0   0  43
       0.8   1.000000     1.0 1.000000                37                43  37   0   0  43
       0.9   1.000000     1.0 1.000000                37                43  37   0   0  43
  best_threshold_by_f1 = 0.2  (f1=1.0000)

## 13. source_dataset별 검증 분석

각 type 별 validation set 을 source_dataset 으로 나눠 별도 분석. threshold=0.5 기준.

- **gt_v2** (allow/block 모두 있음): 일반 classification metrics
- **tickle_human_v2** (모두 ALLOW): false_block_count / false_block_rate / mean_P_BLOCK / P_BLOCK 분위수 (25/50/75/90/95)

In [13]:
source_metrics_by_type: dict[str, dict] = {}

for t in args.TYPES:
    sub_va = train_data[t]["sub_va"]
    y_va = train_data[t]["y_va"]
    p_block = proba_cache[t]
    src_arr = sub_va["source_dataset"].values
    by_source = {}
    print(f"\n========== [{t}] source_dataset 분석 (valid set) ==========")
    for sd in sorted(np.unique(src_arr)):
        mask = src_arr == sd
        y_sd = y_va[mask]
        p_sd = p_block[mask]
        if sd == "tickle_human_v2":
            # 모두 ALLOW 가정. positive=BLOCK 으로 오탐 분석.
            pred_block = (p_sd >= 0.5).astype(int)
            n = int(len(y_sd))
            n_false_block = int(pred_block.sum())
            quantiles = np.quantile(p_sd, [0.25, 0.5, 0.75, 0.90, 0.95]) if n > 0 else np.array([np.nan]*5)
            m = {
                "n_samples": n,
                "n_allow": int((y_sd == 0).sum()),
                "n_block": int((y_sd == 1).sum()),
                "false_block_count": n_false_block,
                "false_block_rate": (n_false_block / n) if n > 0 else None,
                "mean_P_BLOCK": float(p_sd.mean()) if n > 0 else None,
                "P_BLOCK_quantiles": {
                    "q25": float(quantiles[0]), "q50": float(quantiles[1]),
                    "q75": float(quantiles[2]), "q90": float(quantiles[3]),
                    "q95": float(quantiles[4]),
                },
            }
        else:
            if len(y_sd) == 0:
                m = {"n_samples": 0, "note": "empty"}
            else:
                m = compute_basic_metrics(y_sd, p_sd, threshold=0.5)
        by_source[sd] = m

        print(f"\n  [{t} - {sd}]")
        for k, v in m.items():
            print(f"    {k}: {v}")
    source_metrics_by_type[t] = by_source


========== [booking] source_dataset 분석 (valid set) ==========

  [booking - gt_v2]
    threshold: 0.5
    n_samples: 50
    n_allow: 7
    n_block: 43
    accuracy: 1.0
    precision: 1.0
    recall: 1.0
    f1: 1.0
    tn: 7
    fp: 0
    fn: 0
    tp: 43
    pred_allow_count: 7
    pred_block_count: 43
    roc_auc: 1.0

  [booking - tickle_human_v2]
    n_samples: 30
    n_allow: 30
    n_block: 0
    false_block_count: 0
    false_block_rate: 0.0
    mean_P_BLOCK: 0.006976232858910591
    P_BLOCK_quantiles: {'q25': 0.00043463474943356483, 'q50': 0.0009327635679151156, 'q75': 0.003400523890440843, 'q90': 0.015319613553838386, 'q95': 0.02685985109422512}

========== [captcha] source_dataset 분석 (valid set) ==========

  [captcha - gt_v2]
    threshold: 0.5
    n_samples: 54
    n_allow: 6
    n_block: 48
    accuracy: 1.0
    precision: 1.0
    recall: 1.0
    f1: 1.0
    tn: 6
    fp: 0
    fn: 0
    tp: 48
    pred_allow_count: 6
    pred_block_count: 48
    roc_auc: 1.0

  [captcha

## 14. 모델 및 metadata 저장

저장 경로: `../model_joblib/logistic_30_v3/{type}/` 안에 `model.joblib` + `meta.json` + `metrics.json` 3종.
(직전 v1 / logistic_29 / RF28 양식과 동일한 폴더별 3-파일 구조)

In [14]:
if args.SAVE_MODEL:
    saved_paths: dict[str, dict[str, str]] = {}
    for t in args.TYPES:
        sub = df_by_type[t]
        tr = train_data[t]["sub_tr"]
        va = train_data[t]["sub_va"]
        pipe = models[t]
        out_dir = os.path.join(args.MODEL_SAVE_DIR, t)
        os.makedirs(out_dir, exist_ok=True)
        model_path   = os.path.join(out_dir, "model.joblib")
        meta_path    = os.path.join(out_dir, "meta.json")
        metrics_path = os.path.join(out_dir, "metrics.json")

        # 1) model.joblib
        joblib.dump(pipe, model_path)

        # 2) meta.json
        model_name = f"logistic_30_v3_{t}"
        meta = {
            "model_name": model_name,
            "run_version": args.RUN_VERSION,
            "model_type": "LogisticRegression",
            "target_type": t,
            "trained_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
            "label_mapping": {"ALLOW": 0, "BLOCK": 1},
            "positive_class": "BLOCK",
            "positive_class_index": 1,
            "probability_column": "P_BLOCK",
            "decision_rule": "predict BLOCK if P_BLOCK >= threshold",
            "feature_schema_version": "30_v3",
            "n_features": len(BASE_FEATURES_30),
            "feature_names": BASE_FEATURES_30,
            "selected_features_for_training": selected_features_for_training,
            "feature_col_map": feature_col_map,
            "excluded_features": EXCLUDED_FEATURES,
            "data_root": args.DATA_ROOT,
            "included_data_dirs": list(args.INCLUDED_DATA_DIRS),
            "source_dataset_policy": {
                "gt_v2": "label inferred from allow/block directory",
                "tickle_human_v2": "all samples are treated as ALLOW",
            },
            "split_config": {
                "valid_size": args.VALID_SIZE,
                "random_state": args.RANDOM_STATE,
                "split_method": split_info[t]["split_method"],
                "group_key": split_info[t]["group_key"],
            },
            "dataset_summary": {
                "total_samples": int(len(sub)),
                "train_samples": int(len(tr)),
                "valid_samples": int(len(va)),
                "label_distribution_total": sub["label_norm"].value_counts().to_dict(),
                "label_distribution_train": tr["label_norm"].value_counts().to_dict(),
                "label_distribution_valid": va["label_norm"].value_counts().to_dict(),
                "source_label_distribution_total":
                    sub.groupby(["source_dataset", "label_norm"]).size().unstack(fill_value=0).to_dict(),
                "source_label_distribution_train":
                    tr.groupby(["source_dataset", "label_norm"]).size().unstack(fill_value=0).to_dict(),
                "source_label_distribution_valid":
                    va.groupby(["source_dataset", "label_norm"]).size().unstack(fill_value=0).to_dict(),
            },
            "model_params": pipe.named_steps["model"].get_params(),
            "best_threshold_by_f1": float(best_threshold_by_type[t]),
        }
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2, default=str)

        # 3) metrics.json
        metrics = {
            "overall_at_default": overall_metrics_by_type[t],
            "threshold_sweep": threshold_sweep_by_type[t],
            "best_threshold_by_f1": float(best_threshold_by_type[t]),
            "source_dataset_metrics": source_metrics_by_type[t],
        }
        with open(metrics_path, "w", encoding="utf-8") as f:
            json.dump(metrics, f, ensure_ascii=False, indent=2, default=str)

        saved_paths[t] = {
            "model": os.path.abspath(model_path),
            "meta": os.path.abspath(meta_path),
            "metrics": os.path.abspath(metrics_path),
        }
        print(f"[{t}] saved:")
        for k, v in saved_paths[t].items():
            print(f"  {k}: {v}")
else:
    saved_paths = {t: {"model": None, "meta": None, "metrics": None} for t in args.TYPES}
    print("SAVE_MODEL=False — 모델 저장 건너뜀.")

[booking] saved:
  model: c:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\train\model_joblib\logistic_30_v3\booking\model.joblib
  meta: c:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\train\model_joblib\logistic_30_v3\booking\meta.json
  metrics: c:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\train\model_joblib\logistic_30_v3\booking\metrics.json
[captcha] saved:
  model: c:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\train\model_joblib\logistic_30_v3\captcha\model.joblib
  meta: c:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\train\model_joblib\logistic_30_v3\captcha\meta.json
  metrics: c:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\train\model_joblib\logistic_30_v3\captcha\metrics.json
[detail] saved:
  model: c:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\train\model_joblib\logistic_30_v3\detail\model.joblib
  meta: c:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\train\model_joblib\logistic_30_v3\detail\meta.json
  metrics: c:\Us

## 15. 최종 요약 테이블

3개 type 결과를 한 줄씩 요약.

In [15]:
summary_rows = []
for t in args.TYPES:
    sub = df_by_type[t]
    tr = train_data[t]["sub_tr"]
    va = train_data[t]["sub_va"]
    overall = overall_metrics_by_type[t]
    src_m = source_metrics_by_type[t]

    gt = src_m.get("gt_v2", {})
    tk = src_m.get("tickle_human_v2", {})

    allow_count = int((sub["y"] == 0).sum())
    block_count = int((sub["y"] == 1).sum())
    gt_allow = int(((sub["source_dataset"] == "gt_v2") & (sub["y"] == 0)).sum())
    gt_block = int(((sub["source_dataset"] == "gt_v2") & (sub["y"] == 1)).sum())
    tk_allow = int(((sub["source_dataset"] == "tickle_human_v2") & (sub["y"] == 0)).sum())

    summary_rows.append({
        "model_name": f"logistic_30_v3_{t}",
        "target_type": t,
        "n_samples": int(len(sub)),
        "train_samples": int(len(tr)),
        "valid_samples": int(len(va)),
        "allow_count": allow_count,
        "block_count": block_count,
        "gt_v2_allow_count": gt_allow,
        "gt_v2_block_count": gt_block,
        "tickle_human_v2_allow_count": tk_allow,
        "accuracy": overall["accuracy"],
        "precision": overall["precision"],
        "recall": overall["recall"],
        "f1": overall["f1"],
        "roc_auc": overall.get("roc_auc"),
        "best_threshold_by_f1": float(best_threshold_by_type[t]),
        "tickle_false_block_rate": tk.get("false_block_rate"),
        "model_path": saved_paths[t]["model"],
        "metadata_path": saved_paths[t]["meta"],
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

            model_name target_type  n_samples  train_samples  valid_samples  allow_count  block_count  gt_v2_allow_count  gt_v2_block_count  tickle_human_v2_allow_count  accuracy  precision  recall       f1  roc_auc  best_threshold_by_f1  tickle_false_block_rate                                                                                                   model_path                                                                                             metadata_path
logistic_30_v3_booking     booking        401            321             80          181          220                 30                220                          151  1.000000   1.000000     1.0 1.000000 1.000000                   0.2                 0.000000 c:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\train\model_joblib\logistic_30_v3\booking\model.joblib c:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\train\model_joblib\logistic_30_v3\booking\meta.json
logistic_30_v3_captcha     captcha        46